# 01 · Construcción del dataset

Pipeline (se ejecutó localmente con Docker `mysql:8.4.10`; también corre en Colab):

1. **Spider** → `scripts/migrar_spider.py`: migra las 146 bases SQLite de `train_spider` + `train_others` a MySQL 8.4.10 infiriendo tipos de los valores reales.
2. `scripts/construir_spider.py`: transpila cada consulta a MySQL (sqlglot, corrigiendo mayúsculas de tablas), **la ejecuta en MySQL 8.4.10** y descarta las que fallan con su motivo. Toma una muestra estratificada (25 % fácil, 40 % media, 35 % difícil) y traduce las preguntas al español con `Helsinki-NLP/opus-mt-en-es`, protegiendo los literales del SQL.
3. **Esquemas propios** (11) en `datos/esquemas_propios/*.sql`, con `COMMENT`, nombres en español / inglés / abreviados; datos sintéticos deterministas con `scripts/generar_datos.py`. Pares en `datos/pares_propios.jsonl`, generados con LLM y revisados.
4. `scripts/construir_dataset.py`: valida los pares propios, une todo, verifica que **ningún esquema de validación** esté en entrenamiento y separa un 5 % de esquemas para la *eval loss*.

El esquema de cada ejemplo es la salida real de `SHOW CREATE TABLE`, igual que en la evaluación oculta.

In [ ]:
# Obtener el código: subir proyecto_3_deep.zip a /content (panel Archivos) o clonar el repo privado
import os, getpass
REPO = "proyecto_3_deep"
if not os.path.exists(f"/content/{REPO}/solucion.py"):
    if os.path.exists(f"/content/{REPO}.zip"):
        !unzip -q -o /content/{REPO}.zip -d /content/{REPO}
        # si el zip traía la carpeta raíz adentro, subirla un nivel
        if not os.path.exists(f"/content/{REPO}/solucion.py") and os.path.exists(f"/content/{REPO}/{REPO}/solucion.py"):
            !mv /content/{REPO}/{REPO}/* /content/{REPO}/ && rm -rf /content/{REPO}/{REPO}
    else:
        usuario = input("Usuario de GitHub: ")
        token = getpass.getpass("Token de GitHub: ")
        url_repo = input("URL del repo (https://github.com/<owner>/<repo>.git): ").replace("https://", f"https://{usuario}:{token}@")
        !git clone -q {url_repo} /content/{REPO}
%cd /content/{REPO}
!pip install -q -r requirements.txt

In [ ]:
# MySQL 8.4.10 con configuración por defecto (ONLY_FULL_GROUP_BY incluido)
!bash scripts/setup_mysql.sh
import os
os.environ["MYSQL_PORT"] = "3306"
os.environ["MYSQL_PASSWORD"] = "root"

In [ ]:
# (Opcional, ~20 min) Reconstruir todo desde cero. Requiere descargar Spider.
RECONSTRUIR = False
if RECONSTRUIR:
    !pip install -q gdown sentencepiece sacremoses
    !gdown 1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J -O /content/spider.zip && unzip -q -o /content/spider.zip -d /content/spider
    !python scripts/migrar_spider.py /content/spider/spider_data
    !python scripts/construir_spider.py /content/spider/spider_data --muestra 2300
    !python scripts/generar_datos.py datos/esquemas_propios/*.sql
    !python scripts/evaluador.py --preparar
    !python scripts/construir_dataset.py

In [ ]:
import json, pandas as pd
res = json.load(open("datos/resumen_datos.json", encoding="utf-8"))
print(json.dumps({k: v for k, v in res.items() if k != "spider"}, ensure_ascii=False, indent=2))
print(json.dumps(res["spider"], ensure_ascii=False, indent=2))

In [ ]:
desc = pd.read_csv("datos/descartes.csv")
print("Descartes totales:", len(desc))
display(desc.groupby(["fuente", "motivo"]).size().rename("n").reset_index().sort_values("n", ascending=False))

In [ ]:
train = pd.read_json("datos/entrenamiento.jsonl", lines=True)
display(train.groupby(["fuente", "nivel"]).size().unstack(fill_value=0))
display(train.sample(5, random_state=2026)[["db_id", "pregunta", "sql"]])